# Train the from-scratch LLM on a Colab GPU
Repo: `PixelMemory/llm-from-scratch` · saves the checkpoint to your Google Drive.

**To run:** set the runtime to a GPU (Runtime → Change runtime type → A100, if your plan offers it), then **Runtime → Run all**. You'll be asked to authorize Google Drive once.

If the repo is *private*, add a GitHub token to Colab Secrets named `GITHUB_TOKEN` (with notebook access) first — the clone cell uses it. For a public repo, no token is needed.

In [ ]:
# 1) Check the GPU we were assigned
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true
assert torch.cuda.is_available(), 'No CUDA GPU — set Runtime → Change runtime type → GPU'
print('torch', torch.__version__, '| device:', torch.cuda.get_device_name(0))

In [ ]:
# 2) Get the code (works for public OR private repo)
import os, subprocess
REPO = "PixelMemory/llm-from-scratch"
token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")   # only needed if the repo is private
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.isdir("/content/llm-from-scratch"):
    subprocess.run(["git", "clone", "--depth", "1", url], check=True, cwd="/content")
print("cloned:", os.path.isdir("/content/llm-from-scratch"))

In [ ]:
# 3) Fetch a real dataset (tiny-shakespeare ~1MB) — replaces the 4KB sample
%cd /content/llm-from-scratch
!python colab/get_data.py

In [ ]:
# 4) Mount Google Drive (one OAuth click) for checkpoint persistence
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs("/content/drive/MyDrive/llm-from-scratch", exist_ok=True)
print("drive ready")

In [ ]:
# 5) Train on the GPU (bf16 autocast on CUDA). ~a few minutes on an A100.
!mkdir -p out   # ensure the dir exists before tee writes the log
!python train.py --profile gpu --data data/input.txt --iters 6000 2>&1 | tee out/train_gpu.log

In [ ]:
# 6) Copy the checkpoint + log to Drive so they survive the session
import shutil, glob, os
DRIVE = "/content/drive/MyDrive/llm-from-scratch"
shutil.copy("out/ckpt.pt", os.path.join(DRIVE, "ckpt.pt"))
for f in glob.glob("out/*.log"):
    shutil.copy(f, DRIVE)
print("saved to Drive:", os.listdir(DRIVE))

In [ ]:
# 7) Quick sample from the trained model
!python generate.py --impl reference --prompt "ROMEO:" --tokens 400 --temperature 0.8